# Diagnosis to Medication — DX-to-Med Mode

Interactive notebook with **live streaming HTML** that mimics the `/refine_with_kg` route.
Agent reasoning, tool calls, tool results, and recommendations render in real-time
using the same `chat_med_refinement.html` UI.

**Mode:** Given a diagnosis + list of medications, validates which medications are
associated treatments for that diagnosis in the IMO Knowledge Graph.

## 0. Setup

In [ ]:
import sys, os, re, json, html as html_mod, uuid
from typing import List

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from IPython.display import display, HTML, update_display

os.environ["IMO_NORMALIZE_CLIENT_ID"] = ""
os.environ["IMO_NORMALIZE_SECRET"]    = ""
os.environ["IMO_KG_CLIENT_ID"]        = ""
os.environ["IMO_KG_CLIENT_SECRET"]    = ""

os.environ["AWS_ACCESS_KEY_ID"]     = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"]     = ""
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import kg_config as kc
print("\u2713  Credentials loaded")

✓  Credentials loaded


## 1. LLM, Tools & System Prompt

In [2]:
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from kg_api_client import KGApiClient
from typing import Any

llm = ChatBedrockConverse(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0,
    provider="anthropic",
    max_tokens=64000,
)
client = KGApiClient()

# --- Token overflow prevention (from route's _cap_tool_response) ---
_KG_LIST_THRESHOLD = 200
_MAX_TOOL_RESPONSE_CHARS = 100000

def _truncate_kg_lists(data, candidate_problems=None):
    if not isinstance(data, dict):
        return data
    list_fields = [
        "treatedProblems", "causedProblems", "domainNarrower", "narrower",
        "synonyms", "associatedTreatments", "contraindicatedTreatments",
        "supportiveTreatments", "domainNarrowerByMedications",
        "domainNarrowerByProcedures", "containingMedications",
    ]
    result = {}
    for key, value in data.items():
        if key in list_fields and isinstance(value, list) and len(value) > _KG_LIST_THRESHOLD:
            total_count = len(value)
            if candidate_problems:
                candidates_lower = [c.lower().strip() for c in candidate_problems]
                matched = [
                    item for item in value
                    if any(
                        cand in item.get("title", "").lower()
                        or item.get("title", "").lower() in cand
                        for cand in candidates_lower
                    )
                ]
                result[key] = matched
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing " + str(len(matched)) + " items matching candidates. "
                    + "Full list too large (" + str(total_count) + " items)."
                )
            else:
                result[key] = value[:_KG_LIST_THRESHOLD]
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing first " + str(_KG_LIST_THRESHOLD) + " of " + str(total_count) + ". "
                    + "Truncated to prevent token overflow."
                )
        elif key == "lexical" and isinstance(value, dict):
            result[key] = _truncate_kg_lists(value, candidate_problems)
        else:
            result[key] = value
    return result

def _aggressive_truncate(data, max_items):
    if not isinstance(data, dict):
        return data
    result = {}
    for key, value in data.items():
        if isinstance(value, list) and len(value) > max_items:
            total = len(value)
            result[key] = value[:max_items]
            result[key + "_summary"] = (
                "KG returned " + str(total) + " total items. "
                + "Showing first " + str(max_items) + " of " + str(total)
                + " (truncated for token limit)."
            )
        elif isinstance(value, dict):
            result[key] = _aggressive_truncate(value, max_items)
        else:
            result[key] = value
    return result

def _cap_tool_response(result, candidate_problems=None):
    result = _truncate_kg_lists(result, candidate_problems)
    output = json.dumps(result, indent=2)
    if len(output) <= _MAX_TOOL_RESPONSE_CHARS:
        return output
    shrink_limit = 50
    while len(output) > _MAX_TOOL_RESPONSE_CHARS and shrink_limit >= 10:
        result = _aggressive_truncate(result, shrink_limit)
        output = json.dumps(result, indent=2)
        shrink_limit = shrink_limit // 2
    if len(output) > _MAX_TOOL_RESPONSE_CHARS:
        result = _aggressive_truncate(result, 5)
        output = json.dumps(result, indent=2)
    return output

# --- Tool definitions (exact match with refine_with_kg route) ---

@tool
def normalize_medical_term(input_term: str, domain: str = "Medication", order_name: str = "") -> str:
    """Normalize a medical term using IMO Precision Normalize API.

    Args:
        input_term: The clinical term (e.g., "Metformin", "diabetes mellitus", "glucose"). Must not be empty.
        domain: "Medication" for drugs, "Problem" for diagnoses, "Lab" for lab results. Default is Medication.
        order_name: Optional lab order name (e.g., "lipid panel", "comprehensive metabolic panel"). Only used when domain="Lab". If you only have the order name and not the component name, pass the order name as input_term instead.
    """
    if not input_term or not input_term.strip():
        return json.dumps({"success": False, "error": "STOP: input_term is empty. Do NOT retry with empty string. Do NOT call this tool again without a valid medication name. Skip and proceed to scoring."})
    result = client.normalize_medical_term(input_term.strip(), domain, order_name=order_name.strip() if order_name else "")
    return json.dumps(result, indent=2)

@tool
def validate_medication_association(imo_lexical_code: str, medication_codes: str = "") -> str:
    """Check which medications are associated with a diagnosis via KG associatedTreatments.

    Fetches the full associatedTreatments list for the diagnosis and checks which of
    the provided medication codes appear. Reports matched vs unmatched.

    Args:
        imo_lexical_code: The default_lexical_code of the problem (from normalize with domain="Problem"). REQUIRED.
        medication_codes: Comma-separated medication lexical codes to check (e.g., "123456,789012,345678"). REQUIRED.
    """
    if not imo_lexical_code or not imo_lexical_code.strip():
        return json.dumps({"success": False, "error": "STOP: imo_lexical_code is empty. Normalize the diagnosis first to get its default_lexical_code."})
    result = client.get_treatments_for_problem(imo_lexical_code.strip())

    if not result.get("success") or not medication_codes or not medication_codes.strip():
        return _cap_tool_response(result)

    codes_to_check = set(c.strip() for c in medication_codes.split(",") if c.strip())
    lexical = result.get("lexical", {})
    associated = lexical.get("associatedTreatments", [])

    treatments_by_code = {}
    for med in associated:
        treatments_by_code[med.get("code")] = med.get("title", "")

    matched = []
    matched_codes = set()
    for code in codes_to_check:
        if code in treatments_by_code:
            matched.append({
                "medication_code": code,
                "medication_title": treatments_by_code[code],
            })
            matched_codes.add(code)

    unmatched_codes = list(codes_to_check - matched_codes)

    return json.dumps({
        "success": True,
        "diagnosis": {"code": imo_lexical_code, "title": lexical.get("title", "")},
        "kg_summary": {
            "total_associated_treatments": len(associated),
        },
        "matched": matched,
        "unmatched_codes": unmatched_codes,
        "summary": str(len(matched)) + " of " + str(len(codes_to_check)) + " medications found in KG associated treatments for this diagnosis.",
    }, indent=2)

DX_TO_MED_TOOLS = [normalize_medical_term, validate_medication_association]

SYSTEM_PROMPT = """You are a medication validation agent. Given a diagnosis and a list of medications, you query the IMO Knowledge Graph to determine which medications from the user's list are associated treatments for the diagnosis and which are not.

## WORKFLOW

### Step 1: Extract Diagnosis and Medications (TEXT ONLY — NO TOOL CALLS)
Read the user input and identify:
- The diagnosis/problem name
- The full list of medication names

Output a brief summary of what you extracted. Example:
"Extracted diagnosis: Hypertension. Medications: Lisinopril, Metformin, Amlodipine, Atorvastatin, Metoprolol."

DO NOT call any tool during this step. Just extract and summarize.

### Step 2: Normalize the Diagnosis
Call normalize_medical_term(input_term=<diagnosis name>, domain="Problem") to get its default_lexical_code.
- The input_term MUST be the diagnosis name you extracted in Step 1 (e.g., "Hypertension").
- NEVER pass an empty string as input_term.

### Step 3: Normalize All Medications
For EACH medication name extracted in Step 1, call normalize_medical_term with:
- input_term = the medication name (MUST be non-empty, e.g., "Lisinopril")
- domain = "Medication" (NEVER use "Problem" for medications)

CRITICAL RULES FOR THIS STEP:
- ONLY call normalize_medical_term with actual medication names from Step 1.
- NEVER call normalize_medical_term with an empty string "".
- NEVER use domain="Problem" for medication normalization — ALWAYS use domain="Medication".
- If you have 5 medications, you make exactly 5 calls, each with a non-empty medication name.
- Collect all successful default_lexical_codes. Track which input name maps to which code.

### Step 4: Validate Medication Associations
Call validate_medication_association(imo_lexical_code=<diagnosis code from Step 2>, medication_codes=<comma-separated medication codes from Step 3>).

The tool internally:
1. Fetches all associatedTreatments from the KG for this diagnosis
2. Checks which of the provided medication codes appear in that list
3. Returns compact results: matched medications and unmatched codes

### Step 5: Present Results

#### Diagnosis-to-Medication Validation

**Diagnosis:** (title) (`code`)

**Associated Medications (found in KG):**

| Medication (User Input) | Matched KG Title | Medication Code |
|---|---|---|
| (user's term) | (KG title) | (medication code) |

**Not Associated (not found in KG):**

| Medication (User Input) | Normalized Title | IMO Code | Reason |
|---|---|---|---|
| (user's term) | (normalized title) | (code) | Not found in KG associated treatments for this diagnosis |

**Summary:**
- X of Y medications found in KG associated treatments
- Z medications not found in KG for this diagnosis

## ABSOLUTE RULES
- NEVER use your own medical knowledge to determine if a medication treats a diagnosis
- A medication is "associated" ONLY if its normalized IMO code appears in the KG associatedTreatments response — nothing else counts
- A medication is "not associated" if its code does NOT appear in the KG response — even if you know it treats the condition
- If validate_medication_association returns empty treatments, state "No treatments found in KG for this diagnosis" — do NOT infer associations
- If normalize fails for a medication, list it as "Could not normalize — skipped"
- Before each tool call, briefly state what you are normalizing (e.g., "Normalizing diagnosis: X" or "Normalizing medication: Y")
- NEVER call normalize_medical_term with input_term="" — this is a hard error

## SUMMARY BLOCK (REQUIRED)
After your final answer, include a summary block wrapped in `[SUMMARY]` and `[/SUMMARY]` tags. This is displayed in a visual explainer panel for non-technical audiences. Write in simple, plain language.

**FORMAT: Exactly 4 paragraphs separated by blank lines.** Each paragraph maps to a visual card:

1. **Paragraph 1 — Clinical Question:** Restate what the user asked in 1-2 plain sentences. Mention the diagnosis and medications being validated.
2. **Paragraph 2 — What the Agent Did:** Describe the steps: normalized the diagnosis and medications, queried the Knowledge Graph for treatment associations.
3. **Paragraph 3 — Knowledge Graph Value:** The KG provided curated treatment associations for the diagnosis, identifying which medications are clinically linked. Emphasize that these are **curated, clinically-verified relationships** that cannot be derived from text matching — this is the KG's unique value.
4. **Paragraph 4 — Recommendation:** State which medications were validated as associated and which were not.

Keep each paragraph concise (2-4 sentences max). Bold key terms."""

print("✓  LLM, Tools, and System Prompt ready")
print("   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000")
print("   Tools: " + str([t.name for t in DX_TO_MED_TOOLS]))

✓  LLM, Tools, and System Prompt ready
   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000
   Tools: ['normalize_medical_term', 'validate_medication_association']


## 2. HTML Renderer + Streaming Engine

In [3]:
def esc(text):
    return html_mod.escape(str(text)) if text else ""

AGENT_CSS = """
<style>
:root {
    --primary: #57039E; --primary-light: #7C0EDD; --primary-dark: #3D0270;
    --accent: #10b981; --danger: #ef4444;
    --bg: #f3f4f6; --bg-card: #ffffff; --text: #1e293b; --text-secondary: #64748b; --border: #e5e7eb;
}
.agent-panel { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; color:var(--text); line-height:1.6; margin:1rem 0; }
.agent-panel * { box-sizing:border-box; }
.msg-container { display:flex; gap:0.75rem; margin-bottom:1.25rem; animation:fadeIn 0.3s ease-in; }
@keyframes fadeIn { from{opacity:0;transform:translateY(8px)} to{opacity:1;transform:translateY(0)} }
.msg-avatar { width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1.2rem;flex-shrink:0;background:var(--accent); }
.msg-content { flex:1;background:var(--bg-card);border-radius:12px;padding:1rem;box-shadow:0 1px 2px rgba(0,0,0,0.05); }
.msg-header { display:flex;justify-content:space-between;align-items:center;margin-bottom:0.5rem; }
.msg-sender { font-weight:600;color:var(--text);font-size:0.9rem; }
.msg-user .msg-avatar { background:var(--primary); }
.msg-user .msg-content { background:var(--primary);color:#fff; }
.msg-user .msg-sender,.msg-user .msg-time { color:rgba(255,255,255,0.8); }
.reasoning-steps { margin-bottom:1rem;border:1px solid var(--border);border-radius:10px;overflow:hidden;background:var(--bg); }
.reasoning-hdr { display:flex;justify-content:space-between;align-items:center;padding:0.6rem 1rem;background:#f0e7ff;cursor:pointer;user-select:none;border-bottom:1px solid var(--border); }
.reasoning-hdr h4 { font-size:0.88rem;color:var(--primary-dark);margin:0; }
.step-count { font-size:0.68rem;font-weight:600;padding:0.1rem 0.45rem;border-radius:999px;background:var(--primary);color:#fff; }
.toggle-icon { font-size:0.7rem;color:var(--text-secondary);transition:transform 0.2s; }
.reasoning-body { padding:0.6rem;display:flex;flex-direction:column;gap:0.6rem; }
.step { padding:0.7rem 0.85rem;background:var(--bg-card);border-radius:8px;position:relative; }
.step-reasoning { border-left:4px solid #f59e0b;background:linear-gradient(135deg,#fffbeb,var(--bg-card)); }
.step-tool { border-left:4px solid #6d28d9;background:linear-gradient(135deg,#ede9fe,#f5f3ff);box-shadow:0 1px 3px rgba(109,40,217,0.1); }
.step-tool.normalize { border-left-color:#0891b2;background:linear-gradient(135deg,#ecfeff,var(--bg-card));box-shadow:none; }
.step-result { border-left:4px solid var(--accent); }
.step-badge { position:absolute;top:0.5rem;right:0.6rem;color:#fff;font-size:0.63rem;font-weight:700;padding:0.1rem 0.4rem;border-radius:999px;text-transform:uppercase; }
.badge-kg { background:#7c3aed; }
.badge-normalize { background:#0891b2; }
.badge-done { background:var(--accent); }
.step-hdr { display:flex;align-items:center;gap:0.4rem;margin-bottom:0.25rem;font-weight:600;font-size:0.88rem; }
.step-icon { width:22px;height:22px;border-radius:50%;display:inline-flex;align-items:center;justify-content:center;font-size:0.75rem;flex-shrink:0; }
.icon-tool { background:#ede9fe;color:#7c3aed; }
.icon-normalize { background:#cffafe;color:#0e7490; }
.icon-result { background:#d1fae5;color:#065f46; }
.icon-reasoning { background:#fef3c7;color:#92400e; }
.step-detail { font-size:0.83rem;color:var(--text-secondary);padding-left:1.8rem; }
.step-reasoning .step-title { color:#92400e; }
.step-reasoning .reasoning-text { font-size:0.83rem;color:var(--text);line-height:1.5;padding-left:1.8rem; }
.step-tool.normalize .step-title { color:#0e7490; }
.raw-toggle { margin-top:0.5rem;padding-top:0.5rem;border-top:1px solid var(--border); }
.raw-toggle summary { font-size:0.75rem;font-weight:600;color:#7c3aed;cursor:pointer;padding:0.3rem 0.5rem;border-radius:6px; }
.raw-toggle summary:hover { background:rgba(87,3,158,0.05); }
.raw-toggle pre { margin:0.4rem 0 0;background:#f8f9fa;border:1px solid #e2e8f0;border-radius:6px;padding:0.6rem;max-height:300px;overflow-y:auto;font-size:0.72rem;white-space:pre-wrap;word-wrap:break-word; }
.final-answer { margin-top:0.5rem; }
.final-label { font-size:0.73rem;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:var(--accent);margin-bottom:0.3rem;padding-bottom:0.3rem;border-bottom:2px solid var(--accent); }
.final-body { font-size:0.95rem;line-height:1.7; }
.final-body strong { color:var(--primary); }
.final-body code { background:#f1f5f9;padding:0.1rem 0.35rem;border-radius:4px;font-size:0.85em; }
.streaming-indicator { display:flex;align-items:center;gap:0.35rem;padding:0.75rem 1rem;color:var(--primary);font-weight:600;font-size:0.88rem;background:linear-gradient(135deg,rgba(87,3,158,0.04),rgba(124,14,221,0.06));border-radius:8px;margin-top:0.5rem; }
.dot-pulse { display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--primary);animation:pulse 1.4s ease-in-out infinite; }
.dot-pulse:nth-child(2) { animation-delay:0.2s; }
.dot-pulse:nth-child(3) { animation-delay:0.4s; }
@keyframes pulse { 0%,100%{opacity:0.2;transform:scale(0.7)} 50%{opacity:1;transform:scale(1.2)} }
</style>
"""

TOOL_NAMES = {
    "normalize_medical_term": "Normalize Medical Term",
    "validate_medication_association": "GraphQL: Validate Medication Association (KG)",
}

def is_normalize(tool_name):
    return tool_name == "normalize_medical_term"

def fmt_tool_input(tool_name, args):
    if tool_name == "normalize_medical_term":
        order = args.get("order_name", "")
        order_str = ' (order: <strong>' + esc(order) + '</strong>)' if order else ''
        return (
            'Normalizing <strong>\"' + esc(args.get("input_term", ""))
            + '\"</strong> in domain <strong>' + esc(args.get("domain", "Medication")) + '</strong>'
            + order_str
        )
    elif tool_name == "validate_medication_association":
        codes = args.get("medication_codes", "")
        count = len([c for c in codes.split(",") if c.strip()]) if codes else 0
        return (
            'Validating <strong>' + str(count) + '</strong> medication code(s) against KG for diagnosis <strong>'
            + esc(args.get("imo_lexical_code", "")) + '</strong>'
        )
    return '<pre>' + esc(json.dumps(args, indent=2)) + '</pre>'

def _make_table(headers, rows, max_height="250px", col_widths=None):
    n = len(headers)
    if not col_widths:
        col_widths = [str(round(100 / n)) + "%" for _ in range(n)]
    grid_cols = " ".join(col_widths)
    t = '<div style="overflow-x:auto;margin-top:0.3rem;max-height:' + max_height + ';overflow-y:auto;border:1px solid #e5e7eb;border-radius:6px;">'
    t += '<div style="display:grid;grid-template-columns:' + grid_cols + ';background:#57039E;">'
    for h in headers:
        t += '<div style="color:#fff;font-weight:600;text-align:left;padding:0.4rem 0.6rem;white-space:nowrap;font-size:0.8rem;">' + h + '</div>'
    t += '</div>'
    for idx, row in enumerate(rows):
        bg = "#f8fafc" if idx % 2 == 1 else "#ffffff"
        t += '<div style="display:grid;grid-template-columns:' + grid_cols + ';border-bottom:1px solid #e5e7eb;background:' + bg + ';">'
        for cell in row:
            t += '<div style="padding:0.35rem 0.6rem;color:#1e293b;font-size:0.8rem;overflow:hidden;text-overflow:ellipsis;">' + cell + '</div>'
        t += '</div>'
    t += '</div>'
    return t

def fmt_tool_result_html(tool_name, result_str):
    try:
        data = json.loads(result_str) if isinstance(result_str, str) else result_str
    except (json.JSONDecodeError, TypeError):
        return '<pre>' + esc(str(result_str)[:500]) + '</pre>'

    html_out = ""
    if tool_name == "normalize_medical_term" and isinstance(data, dict) and data.get("results"):
        matches = data["results"][0].get("matches", []) if data["results"] else []
        term = data["results"][0].get("input_term", "") if data["results"] else ""
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);margin-bottom:0.3rem;">'
        html_out += str(len(matches)) + ' result(s) for \"<strong>' + esc(term) + '</strong>\"</div>'
        rows = []
        for m in matches[:5]:
            code_val = m.get("default_lexical_code", m.get("lexical_code", ""))
            icd = ", ".join(m.get("icd10_codes", []))
            rows.append([
                '<strong>' + esc(m.get("title", "")) + '</strong>',
                esc(code_val),
                "{:.3f}".format(m.get("score", 0)),
                esc(icd) if icd else "\u2014",
            ])
        html_out += _make_table(["IMO Description", "Lexical Code", "Score", "ICD-10 Codes"], rows,
                                col_widths=["40%", "25%", "15%", "20%"])

    elif tool_name == "validate_medication_association" and isinstance(data, dict) and data.get("success"):
        dx = data.get("diagnosis", {})
        kg = data.get("kg_summary", {})
        matched = data.get("matched", [])
        unmatched = data.get("unmatched_codes", [])
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);">'
        html_out += 'Diagnosis: <strong>' + esc(dx.get("title", "")) + '</strong>'
        html_out += ' \u2014 ' + str(kg.get("total_associated_treatments", 0)) + ' total KG treatments</div>'
        if matched:
            html_out += '<div style="margin-top:0.3rem;"><strong style="color:#10b981;">\u2713 Matched (' + str(len(matched)) + '):</strong></div>'
            rows = []
            for m in matched:
                rows.append([
                    '<strong>' + esc(m.get("medication_title", "")) + '</strong>',
                    esc(m.get("medication_code", "")),
                ])
            html_out += _make_table(["Medication Title", "Code"], rows, col_widths=["65%", "35%"])
        if unmatched:
            html_out += '<div style="margin-top:0.3rem;"><strong style="color:#ef4444;">\u2717 Unmatched (' + str(len(unmatched)) + '):</strong> '
            html_out += esc(", ".join(unmatched)) + '</div>'
        html_out += '<div style="font-size:0.78rem;color:#64748b;margin-top:0.2rem;">' + esc(data.get("summary", "")) + '</div>'

    else:
        html_out += '<pre style="max-height:200px;overflow:auto;font-size:0.72rem;">'
        html_out += esc(json.dumps(data, indent=2)[:1000]) + '</pre>'

    raw_json = json.dumps(data, indent=2)
    html_out += '<details class="raw-toggle"><summary>View Response JSON</summary>'
    html_out += '<pre>' + esc(raw_json[:3000]) + '</pre></details>'
    return html_out

def fmt_markdown_to_html(text):
    text = re.sub(r'\[SUMMARY\][\s\S]*?\[/SUMMARY\]', '', text).strip()
    lines = text.split('\n')
    result = []
    in_table = False
    table_rows = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('|') and stripped.endswith('|'):
            if re.match(r'^\|[\s\-:|]+\|$', stripped):
                continue
            if not in_table:
                in_table = True
                table_rows = []
            table_rows.append([c.strip() for c in stripped.split('|')[1:-1]])
        else:
            if in_table:
                result.append(_build_table(table_rows))
                in_table = False
            result.append(stripped)
    if in_table:
        result.append(_build_table(table_rows))
    f = '\n'.join(result)
    f = re.sub(r'^#### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^## (.+)$', r'<h3>\1</h3>', f, flags=re.M)
    f = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', f)
    f = re.sub(r'\*(.+?)\*', r'<em>\1</em>', f)
    f = re.sub(r'`(.+?)`', r'<code>\1</code>', f)
    f = f.replace('\n\n', '</p><p>').replace('\n', '<br>')
    return '<p>' + f + '</p>'

def _build_table(rows):
    if not rows:
        return ''
    headers = rows[0]
    data_rows = rows[1:]
    n = len(headers)
    if n == 2:
        widths = ["65%", "35%"]
    elif n == 3:
        widths = ["50%", "30%", "20%"]
    elif n == 4:
        widths = ["35%", "25%", "20%", "20%"]
    elif n == 5:
        widths = ["25%", "20%", "20%", "20%", "15%"]
    elif n == 6:
        widths = ["20%", "18%", "15%", "17%", "15%", "15%"]
    else:
        widths = None
    return _make_table(headers, data_rows, "300px", col_widths=widths)

def _streaming_dots(msg="Agent is working..."):
    return (
        '<div class="streaming-indicator">'
        '<span class="dot-pulse"></span><span class="dot-pulse"></span><span class="dot-pulse"></span>'
        '<span style="margin-left:0.3rem;">' + esc(msg) + '</span>'
        '</div>'
    )

def render_agent_run(activity, final_text, label="Medication Validation Results", streaming=False):
    normalize_n = sum(1 for a in activity if a["type"] == "tool_call" and is_normalize(a["tool"]))
    kg_n = sum(1 for a in activity if a["type"] == "tool_call" and not is_normalize(a["tool"]))
    count_parts = []
    if normalize_n:
        count_parts.append(str(normalize_n) + " Normalize")
    if kg_n:
        count_parts.append(str(kg_n) + " KG")
    count_label = " \u00b7 ".join(count_parts) if count_parts else "0 calls"

    norm_idx = 0
    kg_idx = 0
    steps_html = ""
    for a in activity:
        if a["type"] == "reasoning" and a.get("text", "").strip():
            steps_html += (
                '<div class="step step-reasoning">'
                '<div class="step-hdr"><span class="step-icon icon-reasoning">\U0001f9e0</span>'
                '<span class="step-title">Agent Reasoning</span></div>'
                '<div class="reasoning-text">' + fmt_markdown_to_html(a["text"]) + '</div>'
                '</div>'
            )
        elif a["type"] == "tool_call":
            tool_name = a["tool"]
            norm = is_normalize(tool_name)
            if norm:
                norm_idx += 1
                idx = norm_idx
            else:
                kg_idx += 1
                idx = kg_idx
            cls = "step-tool normalize" if norm else "step-tool"
            badge_cls = "badge-normalize" if norm else "badge-kg"
            badge_lbl = "Normalize" if norm else "KG Call"
            icon_cls = "icon-normalize" if norm else "icon-tool"
            icon = "\u2699" if norm else "\U0001f50d"
            display_name = TOOL_NAMES.get(tool_name, tool_name)
            detail = fmt_tool_input(tool_name, a.get("args", {}))
            steps_html += (
                '<div class="step ' + cls + '">'
                '<span class="step-badge ' + badge_cls + '">' + badge_lbl + ' ' + str(idx) + '</span>'
                '<div class="step-hdr"><span class="step-icon ' + icon_cls + '">' + icon + '</span>'
                '<span class="step-title">' + esc(display_name) + '</span>'
                '<span class="step-badge badge-done" style="position:static;margin-left:auto;">Done</span></div>'
                '<div class="step-detail">' + detail + '</div>'
                '</div>'
            )
        elif a["type"] == "tool_result":
            tool_name = a["tool"]
            display_name = TOOL_NAMES.get(tool_name, tool_name)
            result_html = fmt_tool_result_html(tool_name, a.get("result", ""))
            steps_html += (
                '<div class="step step-result">'
                '<div class="step-hdr"><span class="step-icon icon-result">\u2630</span>'
                '<span class="step-title">Results from ' + esc(display_name) + '</span></div>'
                '<div>' + result_html + '</div>'
                '</div>'
            )

    if final_text:
        bottom = (
            '<div class="final-answer">'
            '<div class="final-label">' + esc(label) + '</div>'
            '<div class="final-body">' + fmt_markdown_to_html(final_text) + '</div>'
            '</div>'
        )
    elif streaming:
        bottom = _streaming_dots("Agent is working...")
    else:
        bottom = ""

    if not activity and streaming:
        reasoning_section = _streaming_dots("Agent is thinking...")
    else:
        reasoning_section = (
            '<div class="reasoning-steps">'
            '<div class="reasoning-hdr" onclick="var b=this.parentElement.querySelector('
            "'.reasoning-body');b.style.display=b.style.display==='none'?'flex':'none'" + '">'
            '<h4>Agent Reasoning &amp; API Calls</h4>'
            '<span class="step-count">' + count_label + '</span>'
            '<span class="toggle-icon">\u25bc</span>'
            '</div>'
            '<div class="reasoning-body">' + steps_html + '</div>'
            '</div>'
        )

    return (
        AGENT_CSS
        + '<div class="agent-panel">'
        '<div class="msg-container">'
        '<div class="msg-avatar">\U0001f916</div>'
        '<div class="msg-content">'
        '<div class="msg-header">'
        '<span class="msg-sender">Medication Validation Agent</span>'
        '</div>'
        + reasoning_section
        + bottom
        + '</div></div></div>'
    )

print("\u2713  HTML renderer + streaming engine ready")

✓  HTML renderer + streaming engine ready


## 3. Agent + Streaming Runner

In [4]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools=DX_TO_MED_TOOLS, prompt=SYSTEM_PROMPT)

_conversations = {}
_NOTE_MIN_WORDS = 30

def _extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n".join(parts)
    return str(content) if content else ""

def _build_activity(messages, skip_count=0):
    activity = []
    final_text = ""
    for msg in messages[skip_count:]:
        role = msg.type if hasattr(msg, 'type') else 'unknown'
        if role == 'ai':
            text = _extract_text(msg.content)
            if text.strip():
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    activity.append({"type": "reasoning", "text": text})
                else:
                    final_text = text
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    activity.append({"type": "tool_call", "tool": tc["name"], "args": tc.get("args", {})})
        elif role == 'tool':
            content = msg.content if isinstance(msg.content, str) else json.dumps(msg.content)
            activity.append({"type": "tool_result", "tool": msg.name, "result": content})
    return activity, final_text

def stream_agent(user_message, display_id, session_id="notebook",
                 label="Medication Validation Results", recursion_limit=100):
    update_display(HTML(render_agent_run([], "", label=label, streaming=True)), display_id=display_id)

    session_key = session_id + "_dx-to-med"

    is_new_clinical_note = len(user_message.split()) >= _NOTE_MIN_WORDS
    if is_new_clinical_note and session_key in _conversations and _conversations[session_key]:
        _conversations[session_key] = []

    messages = _conversations.get(session_key, [])
    if len(messages) > 2:
        messages = messages[-2:]
    messages.append({"role": "user", "content": user_message})

    skip_count = len(messages)

    try:
        last_state = None
        for state in agent.stream(
            {"messages": messages},
            stream_mode="values",
            config={"recursion_limit": recursion_limit},
        ):
            last_state = state
            activity, final_text = _build_activity(state["messages"], skip_count=skip_count)
            html = render_agent_run(activity, final_text, label=label, streaming=not bool(final_text))
            update_display(HTML(html), display_id=display_id)

        all_messages = last_state["messages"] if last_state else messages
        activity, final_text = _build_activity(all_messages, skip_count=skip_count)
        update_display(HTML(render_agent_run(activity, final_text, label=label, streaming=False)), display_id=display_id)

        if final_text:
            messages.append({"role": "assistant", "content": final_text})
        _conversations[session_key] = messages

        return activity, final_text, all_messages

    except Exception as e:
        error_msg = "Agent execution failed: " + str(e)
        print("  \u2717 " + error_msg)
        import traceback
        traceback.print_exc()
        error_html = (
            '<div style="padding:1rem;background:#fef2f2;border:1px solid #fecaca;border-radius:8px;'
            'color:#991b1b;font-size:0.9rem;margin-top:0.5rem;">'
            '<strong>Error:</strong> ' + esc(error_msg) + '</div>'
        )
        update_display(HTML(AGENT_CSS + error_html), display_id=display_id)
        return [], "", messages

print("\u2713  Agent + streaming runner ready (with conversation history management)")

✓  Agent + streaming runner ready (with conversation history management)


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_18360\992257856.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=DX_TO_MED_TOOLS, prompt=SYSTEM_PROMPT)


## 4. Clinical Scenario — Hypertension Medication Validation

In [5]:
CLINICAL_NOTE = """Diagnosis: Hypertension
Medications: Lisinopril, Metformin, Amlodipine, Atorvastatin, Metoprolol"""

user_html = (
    AGENT_CSS
    + '<div class="agent-panel"><div class="msg-container msg-user">'
    + '<div class="msg-avatar">\U0001f464</div>'
    + '<div class="msg-content">'
    + '<div class="msg-header"><span class="msg-sender">You</span></div>'
    + '<div style="font-size:1.05rem;white-space:pre-wrap;">' + esc(CLINICAL_NOTE) + '</div>'
    + '</div></div></div>'
)
display(HTML(user_html))

## 5. Run Agent — Diagnosis to Medication Validation (Streaming)

1. **Extract** diagnosis and medication list
2. **Normalize** diagnosis (domain="Problem")
3. **Normalize** each medication (domain="Medication")
4. **Validate** medication codes against KG associatedTreatments
5. **Present** associated vs not-associated results

In [6]:
stream_id = "dxtomed_" + uuid.uuid4().hex[:6]
display(HTML(render_agent_run([], "", label="Medication Validation Results", streaming=True)), display_id=stream_id)

activity, final_text, all_messages = stream_agent(
    CLINICAL_NOTE,
    display_id=stream_id,
    session_id="notebook",
    label="Medication Validation Results",
    recursion_limit=100,
)

tc_count = sum(1 for a in activity if a['type'] == 'tool_call')
rs_count = sum(1 for a in activity if a['type'] == 'reasoning')
print("\u2713 Complete. " + str(tc_count) + " tool calls, " + str(rs_count) + " reasoning steps.")

✓ Complete. 7 tool calls, 3 reasoning steps.
